# `metrics.py` Reference

Three metrics evaluate a finished tournament against latent `skill`
(which pairing algorithms never saw):

- **`mean_skill_gap`** — avg absolute skill diff per match (lower = better matched)
- **`standings_skill_correlation`** — Pearson correlation of final rank vs skill rank (+1 = perfect)
- **`rematch_count`** — extra meetings beyond the first (lower = fairer)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(
    Path.cwd().parent.parent
    if Path.cwd().name == 'reference'
    else Path.cwd().parent
))

from random import Random
print('ready')

In [ ]:
from tournament.generators import make_players, skilled_match, random_match
from tournament.engine import run_tournament
from tournament.pairing import get as get_pairing

rng = Random(42)
players = make_players(8, rng)
tour = run_tournament(players, n_rounds=4, pairing=get_pairing('adjacent'), rng=rng)

## Individual metrics

In [ ]:
from tournament.metrics import mean_skill_gap, standings_skill_correlation, rematch_count

print(f'mean_skill_gap              = {mean_skill_gap(tour):.4f}')
print(f'standings_skill_correlation = {standings_skill_correlation(tour):.4f}')
print(f'rematch_count               = {rematch_count(tour)}')

## `summary` — all metrics at once

In [ ]:
from tournament.metrics import summary

s = summary(tour)
for k, v in s.items():
    print(f'  {k:<30} = {v}')

## Strategy comparison — 6-round tournament

In [ ]:
rng_base = Random(42)
players  = make_players(8, rng_base)

strategies = ('adjacent', 'fold', 'strong_weak', 'random_within_record', 'random')
print(f'{"Strategy":<22} {"skill_gap":>10} {"correlation":>12} {"rematches":>10}')
print('-' * 58)
for name in strategies:
    t = run_tournament(players, n_rounds=6, pairing=get_pairing(name), rng=Random(42))
    print(f'{name:<22} {mean_skill_gap(t):>10.4f} '
          f'{standings_skill_correlation(t):>12.4f} {rematch_count(t):>10}')

## Effect of round count on correlation

In [ ]:
rng_base = Random(42)
players  = make_players(8, rng_base)
print(f'{"rounds":<8} {"adjacent corr":>16} {"fold corr":>12}')
print('-' * 40)
for n_r in (2, 3, 4, 5, 6, 7):
    t_adj  = run_tournament(players, n_rounds=n_r, pairing=get_pairing('adjacent'),  rng=Random(42))
    t_fold = run_tournament(players, n_rounds=n_r, pairing=get_pairing('fold'),       rng=Random(42))
    print(f'{n_r:<8} {standings_skill_correlation(t_adj):>16.4f} '
          f'{standings_skill_correlation(t_fold):>12.4f}')